<a href="https://colab.research.google.com/github/kaxmarc/statistics_ml/blob/main/group_exercises/exercise_2/Group_Exercise_2_PART_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Group Exercise 2 - Chapter 2 - Regression Analysis and Feature Selection
# PART 2: Build and Evaluate Models

Group consisting of Ritu, Vikrant Singh and Kay Müller

Dataset **"House Prices - Advanced Regression Techniques"**

The dataset contains properties with various environmental variables.

Source:
https://www.kaggle.com/datasets/rishitaverma02/house-prices-advanced-regression-techniques

In [ ]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Lasso, Ridge, LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.feature_selection import RFE
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.tree import DecisionTreeRegressor

##Preparation for: Train a linear regression model

In [ ]:
# Load Kaggle dataset
df = pd.read_csv('https://raw.githubusercontent.com/kaxmarc/statistics_ml/refs/heads/main/datasets/train_house.csv') # RAW Data from GitHub
#house_data.head()
print("Shape:", df.shape)
#print("\nMissing values:")
#print(df.isnull().sum().sort_values(ascending=False))

In [ ]:
from pandas.core.arrays import categorical
numerical_columns = df.select_dtypes(include='number').columns
#print(numerical_columns)
categorical_columns = df.select_dtypes(exclude='number').columns
#print(categorical_columns)

# numerical imputation
imput_median = SimpleImputer(strategy="median")
imput_mean = SimpleImputer(strategy="mean")

for feature in numerical_columns:

  if df[feature].isnull().sum().mean() == 0:  # if none are missing
    continue                                  # continue in the loop

  if df[feature].skew() > 1 or df[feature].skew() < -1:
    # imput_median
    df[[feature]] = imput_median.fit_transform(df[[feature]])
  else:
    # imput_mean
    df[[feature]] = imput_mean.fit_transform(df[[feature]])


# categorical imputation
imputer_mode = SimpleImputer(strategy="most_frequent")

for feature in categorical_columns:
  if df[feature].isnull().sum() == 0:
    continue

  df[[feature]] = imputer_mode.fit_transform(df[[feature]])


# Adjusted features
#print("\nAdjusted features:")
#print(df.isnull().sum().sort_values(ascending=False))


In [ ]:
features = ['YearBuilt', 'SalePrice']
X = df[features].copy()

# Standardisation Z-Score
z_scaler = StandardScaler()
z_X = z_scaler.fit_transform(X)
z_X = pd.DataFrame(z_X, columns=[f + '_z' for f in features])

# Normalisation (Min Max)
mm_scaler = MinMaxScaler()
mm_X = mm_scaler.fit_transform(X)
mm_X = pd.DataFrame(mm_X, columns=[f + '_mm' for f in features])

# compared
comparsion = pd.concat([X, z_X, mm_X], axis=1)
print(comparsion.head())

#Build and Evaluate Models

1.Train a Linear Regression model.

2.Evaluate using R² Score and RMSE.

3.Perform K-Fold Cross-Validation (e.g., 5-fold).

In [48]:
# Feature grouping (splitting)
numerical_cols = ['YearBuilt', 'LotArea', 'OverallQual', 'GrLivArea', 'GarageCars', 'GarageArea']
categorical_cols = ['Neighborhood']

X = df[numerical_cols + categorical_cols]                                         # influencing factors (if necessary further scaling required)
y = df['SalePrice']                                                               # target value

# Train-Test-Splitt
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)                                                                                 # 80% Trainingset and 20% Testset

# One-Hot-Encoder (Categorical cols), StandardScaler
preprocessor = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', numerical_cols),                                   # Scaling mean close to 0 and standard deviation close to 1
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)         # One-Hot-Encoder on 'Neighborhood'
    ])

# Process pipeline (Preparation + Lineare Regression)
model = Pipeline(steps=[
  ('preprocess', preprocessor),
  ('linreg', LinearRegression())
])

# TRAINING: Model training
model.fit(X_train, y_train)

# EVALUATION TRAIN AND TEST DATA ON OVERFITTING
y_train_predict = model.predict(X_train)
y_test_predict = model.predict(X_test)

# Roor Mean Squared Error (forecast quality)
rmse_train = np.sqrt(mean_squared_error(y_train, y_train_predict))
rmse_test = np.sqrt(mean_squared_error(y_test, y_test_predict))


# R2 (Explanatory power of features)
r2_train = r2_score(y_train, y_train_predict)
r2_test = r2_score(y_test, y_test_predict)

print(f"Train RMSE: {rmse_train:,.0f}")
print(f"Test RMSE: {rmse_test:,.0f}")
print(f"Train R2: {r2_train:.2f}")
print(f"Test R2: {r2_test:.2f}")

# K-Fold Cross-Validation 5-fold
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

# R2 mittels Cross-Validation
cv_r2 = cross_val_score(model, X, y, cv=kfold, scoring='r2')

# RMSE mittels Cross-Validation
cv_mse = cross_val_score(model, X, y, cv=kfold, scoring='neg_mean_squared_error')
cv_rmse = np.sqrt(-cv_mse)

print("\n5-fold Cross-Validation:")
print("\nCV-R2 per fold:          ", np.round(cv_r2, 3))
print("\n--> Average CV-R2:       ", cv_r2.mean().round(3), "<--")
print("\nCV-RMSE per fold:        ", np.round(cv_rmse, 0))
print("\n--> Average CV-RMSE:     ", np.round(cv_rmse.mean(), 0), "<--")



Train RMSE: 34,525
Test RMSE: 37,546
Train R2: 0.80
Test R2: 0.82

5-fold Cross-Validation:

CV-R2 per fold:           [0.816 0.821 0.59  0.809 0.839]

--> Average CV-R2:        0.775 <--

CV-RMSE per fold:         [37546. 34916. 47608. 34587. 28998.]

--> Average CV-RMSE:      36731.0 <--


RMSE indicates that this model is off by an average of approximately 37,546 monetary units.

R2 indicates that 82% of the variance in price (the dependent variable) is explained. More precisely:


*   Property prices fluctuate in the dataset (variance)
*   The model used can explain 82% of these fluctuations using the features.
*   18% are unexplained, i.e. errors, random values or missing features. E.g. Is there a special swimming pool, school quality in the region, level of education...

CV-R2, on the other hand, indicates that with **77.5% variance across different training-test splits**, the model generalises stably.

CV-RMSE indicates that the model has a deviation of 36,731 monetary units - **across multiple splits**.




#Enhance the Model
1.Experiment with feature selection techniques and analyze their impact.

2.Visualize results (e.g., feature importance, residual plots).

In [50]:
X = df[numerical_cols + categorical_cols].drop(columns["SalePrice"])
y = df['SalePrice']

categorical_columns = X.select_dtypes(include=["object"]).columns
numerical_columns = X.select_dtypes(exclude=["object"]).columns

num_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

cat_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ohe", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", num_transformer, numerical_columns),
        ("cat", cat_transformer, categorical_columns)
    ],
    remainder="drop"
)

model = DecisionTreeRegressor(
    max_depth = 5,
    min_samples_leaf = 10,
    random_state = 42
)

c_pipe = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", model)
])

# Train-Test-Splitt
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

c_pipe.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Mean Error:", mean_absolute_error(y_test, y_pred))
print("R2 Score:", r2_score(y_test, y_pred))



NameError: name 'columns' is not defined